In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"  # Ensure Keras 3 uses JAX backend


In [ ]:
import keras
import keras_hub
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load Flowers dataset
dataset_name = "tf_flowers"
dataset, dataset_info = tfds.load(
    dataset_name,
    as_supervised=True,
    with_info=True
)
data_train, data_test = dataset["train"].take(2000), dataset["train"].skip(2000).take(1000)  # Faster training with a smaller dataset

BATCH_SIZE = 64  # Increase batch size for faster training
IMAGE_SIZE = (224, 224)
NUM_CLASSES = dataset_info.features['label'].num_classes

def preprocess_inputs(image, label):
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0  # Normalize
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

# Preprocess dataset
data_train = data_train.map(preprocess_inputs).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
data_test = data_test.map(preprocess_inputs).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Load EfficientNetB0 model using keras.applications
base_model = keras.applications.EfficientNetB0(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze the base model

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Build model with additional layers
inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)  # Ensure proper shape before Dense layers
x = keras.layers.Dense(128, activation="relu")(x)  # Reduced layer size for speed
x = keras.layers.Dropout(0.3)(x)  # Lower dropout for speed
x = keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, x)

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),  # Higher LR for faster convergence
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# Train the model
history = model.fit(data_train, validation_data=data_test, epochs=5)  # Reduce epochs for speed

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1608s 50s/step - accuracy: 0.2032 - loss: 1.7548 - val_accuracy: 0.2660 - val_loss: 1.6008
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1562s 49s/step - accuracy: 0.2444 - loss: 1.6007 - val_accuracy: 0.2660 - val_loss: 1.6046
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1571s 50s/step - accuracy: 0.2470 - loss: 1.6053 - val_accuracy: 0.2660 - val_loss: 1.5924
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1549s 49s/step - accuracy: 0.2486 - loss: 1.6058 - val_accuracy: 0.2660 - val_loss: 1.5945
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1565s 49s/step - accuracy: 0.2486 - loss: 1.5997 - val_accuracy: 0.2660 - val_loss: 1.5970


In [ ]:
# Plot accuracy
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()